# TSFS12 Hand-in exercise 4: Collaborative Control 5.1

In [15]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [16]:
%matplotlib qt

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [17]:
def double_integrator(t, x, u, model_params):
    '''
    System-model functions
     
     Input arguments are:
     t - time 
     x - agent state
     u - the control input u, 
     model_params - structure with parameters of the model '''

    velocity = x[2:4]
    return np.concatenate((velocity, u))

def controller_p_v(y, y_ref, ctrl_params):
    ''' 
    Control function   

    Output: Control signal 
    '''
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    p, v = y[:2], y[2:]
    p_ref, v_ref = y_ref[:2], y_ref[2:]

    return Kp * (p_ref - p) + Kv * (v_ref - v)

def controller_p_v_flipped(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    p, v = y[:2], y[2:]
    p_ref, v_ref = y_ref[:2], y_ref[2:]

    return Kp * (p - p_ref) + Kv * (v - v_ref)


def controller_relative(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    u = np.zeros(2)

    for k in range(len(y)):
        dp = y[k, :2]
        dv = y[k, 2:4]

        dp_ref = y_ref[k, :2]
        dv_ref = y_ref[k, 2:4]

        u += Kp * (dp - dp_ref) + Kv * (dv - dv_ref)

    return u

def controller_relative_flipped(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    u = np.zeros(2)

    for k in range(len(y)):
        dp = y[k, :2]
        dv = y[k, 2:4]

        dp_ref = y_ref[k, :2]
        dv_ref = y_ref[k, 2:4]

        u += Kp * (dp_ref - dp) + Kv * (dv_ref - dv)

    return u

Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [18]:
n = 4  # (px, py, vx, vy)
G = [ #(i, j) betyder att agent j mäter agent i
    (0,),       # Agent 0 measures its own absolute state

    (0, 1),     # Agent 1 measures agent 0
     

    (1, 2),     # Agent 2 measures agent 1
       

    (2, 3),
  

    (3, 4),
    

    (4, 5),
    

    (5, 6),
    (6, 7),
  

]

def make_formation_references(omega):
    return [
        lambda t: np.array([
            0,
            4 + 0.2 * np.sin(omega * t),
            0,
            0.2 * omega * np.cos(omega * t)
        ]),

        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
        lambda t: np.array([[0, 1, 0, 0]]),
    ]



Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [19]:
# Controller parameters
ctrl_params = {
    "kp": 2.0,
    "kv": 2.0,
}

# Create agents
agents = []

# Agent 0 has absolute measurement
agents.append(
    CreateAgent(
        double_integrator,
        {},
        controller_p_v,
        ctrl_params
    )
)

# Agents 1-7 use relative measurements
for _ in range(7):
    agents.append(
        CreateAgent(
            double_integrator,
            {},
            controller_relative,
            ctrl_params
        )
    )

agents_flipped = []

# Agent 0 with flipped absolute controller
agents_flipped.append(
    CreateAgent(
        double_integrator,
        {},
        controller_p_v_flipped,
        ctrl_params
    )
)

# Agents 1-7 with flipped relative controller
for _ in range(7):
    agents_flipped.append(
        CreateAgent(
            double_integrator,
            {},
            controller_relative_flipped,
            ctrl_params
        )
    )

# Create complete formation
omega = 2.0
formation_references = make_formation_references(omega)

formation = AgentFormation(
    agents,
    G,
    formation_references,
    n
)

x0 = np.array([
    1, 0, 0, 0,
    2, 0, 0, 0,
    3, 0, 0, 0,
    4, 0, 0, 0,
    5, 0, 0, 0,
    6, 0, 0, 0,
    7, 0, 0, 0,
    8, 0, 0, 0,
])

Simulate the formation

In [20]:
t = np.arange(0, 30, 0.05) # Time vector for the simulation
refs_1 = make_formation_references(1.0)
formation_1 = AgentFormation(agents, G, refs_1, n)
x_omega1 = odeint(formation_1.ode, x0, t)

refs_2 = make_formation_references(2.0)
formation_2 = AgentFormation(agents, G, refs_2, n)
x_omega2 = odeint(formation_2.ode, x0, t)

omega_test = 1.0
refs_flip = make_formation_references(omega_test)

formation_flip = AgentFormation(
    agents_flipped,
    G,
    refs_flip,
    n
)

# Shorter simulation because the system may diverge quickly
t_flip = np.arange(0, 10, 0.01)

x_flip = odeint(
    formation_flip.ode,
    x0,
    t_flip
)



Animate the results

In [21]:
fig, ax = plt.subplots(num=20, clear=True, figsize=(8, 6))
ax.axis([-2, 9, -4, 5])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Exercise 5.2.2: Displacement-based formation control")

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:cyan", "tab:purple", "tab:brown", "tab:pink"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
params_text = ax.text(
    0.02, 0.02,
    f"Kp = {ctrl_params['kp']}, Kv = {ctrl_params['kv']}\n",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
)

def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x_omega1[: i + 1, n * idx]
        py = x_omega1[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text, params_text)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x_omega1.shape[0], blit=True, repeat=False
)
xf = x_omega1[-1].reshape(8, 4)

"""
# Plot omega velocity
fig, ax = plt.subplots(figsize=(9, 6))

for agent in range(8):
    vy = x_omega1[:, 4 * agent + 3]
    ax.plot(t, vy, label=f"Agent {agent + 1}")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Vertical velocity [m/s]")
ax.set_title("Vertical velocities, ω = 1 rad/s")
ax.grid()
ax.legend()
plt.show()


# Plot omega 2
fig, ax = plt.subplots(figsize=(9, 6))

for agent in range(8):
    vy = x_omega2[:, 4 * agent + 3]
    ax.plot(t, vy, label=f"Agent {agent + 1}")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Vertical velocity [m/s]")
ax.set_title("Vertical velocities, ω = 2 rad/s")
ax.grid()
ax.legend()
plt.show()
"""

fig, ax = plt.subplots(figsize=(8, 6))

for agent in range(8):
    px = x_flip[:, n * agent]
    py = x_flip[:, n * agent + 1]

    ax.plot(px, py, label=f"Agent {agent + 1}")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Flipped control-law signs")
ax.grid()
ax.legend()
plt.show()